# Solution F2 · Six ways to encode `occupation` (40 levels), with two models

**Type**: Compare

## Objective

Compare six strategies for a high-cardinality categorical variable (none, one-hot, one-hot with rare categories
grouped, ordinal, frequency, out-of-fold target encoding) with a logistic regression and a gradient boosting, in cross-validation.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from common.transformers import FrequencyEncoder, TargetEncoderOOF

df = ds.loans(with_missing=False)
NUM = ["age", "income", "job_tenure", "amount"]
NOM, OCC = ["contract_type"], ["occupation"]
X = df[NUM + NOM + OCC]
y = df["defaulted"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)


def build(occ_encoder, model):
    parts = [("num", StandardScaler(), NUM), ("nom", OneHotEncoder(handle_unknown="ignore"), NOM)]
    if occ_encoder is not None:
        parts.append(("occ", occ_encoder, OCC))
    return make_pipeline(ColumnTransformer(parts, sparse_threshold=0), model)


MODELS = {"logistic": lambda: LogisticRegression(max_iter=2000),
          "boosting": lambda: HistGradientBoostingClassifier(max_iter=100, random_state=0)}
print("occupations with fewer than 20 rows:", int((X.occupation.value_counts() < 20).sum()))

## Commented solution

In [ ]:
encoders = {
    "no occupation": None,
    "one-hot": OneHotEncoder(handle_unknown="ignore"),
    "one-hot rare grouped": OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30),
    "ordinal": OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
    "frequency": FrequencyEncoder(),
    "target OOF": TargetEncoderOOF(m=10),
}

In [ ]:
table = pd.DataFrame(index=list(encoders), columns=list(MODELS), dtype=float)
for enc_name, enc in encoders.items():
    for model_name, make in MODELS.items():
        table.loc[enc_name, model_name] = cross_val_score(build(enc, make()), X, y, cv=cv, scoring="roc_auc").mean()
table.round(3)

In [ ]:
best_logistic = table["logistic"].idxmax()
best_boosting = table["boosting"].idxmax()
logistic_te_gain = table.loc["target OOF", "logistic"] - table.loc["no occupation", "logistic"]
print("best for the logistic:", best_logistic, "| for boosting:", best_boosting)
print(f"gain of target encoding on the logistic: {logistic_te_gain:+.3f} of AUC")

## Check

In [ ]:
with verify():
    check(list(encoders) == ["no occupation", "one-hot", "one-hot rare grouped", "ordinal", "frequency", "target OOF"] and encoders["no occupation"] is None,
          "six encoders, the first one is None", "respect the order and the names required")
    check(table.shape == (6, 2) and table.notna().all().all(), "complete 6 x 2 table", "rows: encoders, columns: logistic, boosting")
    check(table.loc["ordinal", "logistic"] < table.loc["one-hot", "logistic"], "with the logistic the ordinal is worse than the one-hot",
          "a linear model cannot use an integer code with no order")
    check(table.loc["target OOF", "logistic"] > table.loc["no occupation", "logistic"] + 0.005,
          "out-of-fold target encoding adds useful information for the logistic", "target encoding should improve the AUC compared to not using the occupation")
    check(best_logistic == table["logistic"].idxmax() and best_boosting == table["boosting"].idxmax(),
          "best encoders consistent with the table", "use idxmax() on the columns of the table")
    check(close(logistic_te_gain, table.loc["target OOF", "logistic"] - table.loc["no occupation", "logistic"]),
          "target encoding gain computed", "gain = AUC(target OOF) - AUC(no occupation)")
    check(best_logistic != "ordinal" and best_boosting != "no occupation", "the best encoders are not the two trivial cases", "review the table")
done()

## What you learned

- With the **logistic** ordinal (0.786) and frequency (0.786) add nothing compared to dropping the column (0.787): a numeric
  code imposes a linear effect across occupations that have no order. One-hot (0.805), one-hot with rare grouped
  (0.803) and target encoding (0.802) give every occupation its own weight: about +0.02 of AUC.
- With **boosting** the **ordinal** wins (0.771 against 0.760 for the one-hot and 0.744 for target encoding): trees isolate a
  category with two splits, even from a code with no order, and target encoding compresses the occupation into a single number
  that the trees cannot decompose again. The best encoder **depends on the model**.
- Among one-hot, rare grouped and target encoding with the logistic the differences (0.802-0.805) are in the noise: there is no clear
  winner. Target encoding has the advantage of producing a single column (with 500 categories it is decisive).
- Target encoding works **only** if it is out-of-fold and inside the pipeline (exercises A3 and E3); here the pipeline guarantees it.

## Common pitfall

Choosing the encoder on the AUC of a single model, or on the value of a single fold. The differences between encoders are often of the order
of 0.01: they must be read with the standard deviation across folds and repeated with different seeds before concluding.

## Interview question

> *Click on a question to show its answer.*

<details><summary>For a variable with 500 categories and a gradient boosting model, which encoding do you use?</summary>

I would start from the native handling of categoricals (LightGBM, CatBoost, or `HistGradientBoostingClassifier` with `categorical_features`),
which avoids the explosion of columns and does not require choosing an order. Alternatively out-of-fold target encoding with smoothing, or
frequency. One-hot with 500 columns slows things down and fragments the data; ordinal works with trees but with no quality guarantees.
I would compare the options in cross-validation inside the pipeline.

</details>